# AIMS Senegal — Road Traffic Object Detection & Tracking
**Project 2 — Computer Vision | Deadline: April 28, 2026**

This notebook orchestrates the full pipeline: detection, tracking, and counting of road traffic objects across multiple video scenes.
All steps are called via CLI commands from this notebook.

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics kagglehub filterpy flask pandas plotly seaborn tqdm loguru

## 2. Clone Project from GitHub

In [ ]:
!git clone https://github.com/ioget/traffic-rookie-aims-cv.git
%cd traffic-rookie-aims-cv

## 3. Download BDD100K Dataset

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("a7madmostafa/bdd100k-yolo")

DATASET_PATH = path
original_yaml = f"{DATASET_PATH}/data.yaml"

# data.yaml has wrong paths — fix them and write to /tmp/ (always writable)
with open(original_yaml, 'r') as f:
    content = f.read()

content = content.replace('/kaggle/input/bdd100k-yolo', DATASET_PATH)

DATASET_YAML = "/tmp/data_fixed.yaml"
with open(DATASET_YAML, 'w') as f:
    f.write(content)

os.environ['DATASET_PATH'] = DATASET_PATH
os.environ['DATASET_YAML'] = DATASET_YAML

print("Dataset path :", DATASET_PATH)
print("Fixed YAML   :", DATASET_YAML)
print("\nFixed YAML content:")
print(content)

In [ ]:
# Explore dataset structure
!ls -lh {path}
!find {path} -name '*.yaml' | head -5

## 4. Environment Check (GPU, Files)

In [ ]:
!python -c "import torch; print('GPU available:', torch.cuda.is_available()); print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!python -c "from ultralytics import YOLO; print('Ultralytics OK')"
!ls data/
!ls models/

## 5. Fine-tune YOLO on BDD100K

In [ ]:
from src.detection.yolo_detector import YOLODetector

detector = YOLODetector(model_path='models/yolo11n.pt', confidence_threshold=0.5)
detector.fine_tune_model(dataset_config=DATASET_YAML, epochs=10)
print('Fine-tuning complete.')

## 5b. Save Fine-tuned Model

In [ ]:
import shutil
import glob
import os

# Find the best weights produced by fine-tuning
best_weights = glob.glob("runs/detect/**/best.pt", recursive=True)

if best_weights:
    src = best_weights[0]
    dst = "models/yolo11n_finetuned.pt"
    shutil.copy(src, dst)
    print(f"Saved fine-tuned model to {dst}")
    print(f"File size: {os.path.getsize(dst) / 1e6:.1f} MB")
else:
    print("No best.pt found. Make sure fine-tuning completed successfully.")

In [ ]:
# Push fine-tuned model to GitHub
!git config user.email "mamakemrosly@gmail.com"
!git config user.name "ioget"
!git add models/yolo11n_finetuned.pt
!git commit -m "Add fine-tuned YOLOv11 model on BDD100K (10 epochs)"
!git push

## 6. Detection + Tracking — Video 1 (traffic.mp4)

In [ ]:
!python main.py \
    --video data/traffic.mp4 \
    --model models/yolo11n_finetuned.pt \
    --output outputs/traffic_finetuned.avi \
    --confidence 0.5 \
    --tracking

## 7. Detection + Tracking — Video 2 (traffic-sign-test.mp4)

In [ ]:
!python main.py \
    --video data/traffic-sign-test.mp4 \
    --model models/yolo11n_finetuned.pt \
    --output outputs/traffic_sign_finetuned.avi \
    --confidence 0.5 \
    --tracking

## 8. Check Generated Logs

In [ ]:
import json, glob

!ls -lh logs/

log_files = glob.glob('logs/*.json')
for f in log_files:
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    print(f"--- {f} ---")
    print(f"  Total entries: {len(detections)}")
    if detections:
        print(f"  Sample: {detections[0]}")

## 9. Aggregated Statistics

In [ ]:
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    all_detections.extend(detections)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)
print('Total count per class:')
for cls, cnt in class_counts.most_common():
    print(f'  {cls}: {cnt}')
print(f'Total detections: {len(all_detections)}')

## 10. Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    detections = data.get('detections', data) if isinstance(data, dict) else data
    all_detections.extend(detections)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)

if class_counts:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(class_counts.keys(), class_counts.values(), color='steelblue')
    ax.set_title('Detection Count per Object Class')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Detections')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('outputs/stats_classes.png', dpi=150)
    plt.show()
    print('Chart saved to outputs/stats_classes.png')
else:
    print('No log data found. Run steps 6 and 7 first.')

## 11. Web Interface (local use only)
> On Kaggle, the web interface is not directly accessible. Run this locally instead.

In [ ]:
# Uncomment below to launch locally
# !python main.py --video data/traffic.mp4 --model models/yolo11n.pt --web
print("To launch the web interface locally, run: python main.py --video data/traffic.mp4 --web")

## 12. Check Output Files

In [ ]:
!ls -lh outputs/ 2>/dev/null || echo 'outputs/ folder is empty or does not exist'
!ls -lh logs/ 2>/dev/null || echo 'logs/ folder is empty or does not exist'

## 13. Play Output Video

In [ ]:
from IPython.display import Video, display
import glob

output_files = glob.glob("outputs/*.avi") + glob.glob("outputs/*.mp4")

if output_files:
    for f in output_files:
        print(f"Playing: {f}")
        display(Video(f, embed=True, width=800))
else:
    print("No output videos found. Run steps 6 and 7 first.")